In [ ]:
import sys
import os
import warnings
import threading
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
from statsforecast import StatsForecast
from statsforecast.models import AutoTheta, AutoETS

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120, 'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# ── Discord Config ────────────────────────────────────────────
DISCORD_WEBHOOK = "https://discord.com/api/webhooks/1512351711933759641/Fx6qmVuxBlJF2VfOc6IWbrxFI9y3-XR_jDUIayTTFC887IDMDWI0shJ-jJDGCk_f4TRy"

def send_discord(message):
    try:
        payload = {
            'content': message
        }
        requests.post(
            DISCORD_WEBHOOK,
            json=payload,
            timeout=10
        )
    except Exception as e:
        print(f'Discord error: {e}')

print('✓ Imports loaded')
print('✓ Crypto Engine v2.0 (5m entries + Discord)')
print('Supported: BTC, ETH, BNB, SOL, XRP')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 2 — CONFIG (Edit only this cell)            ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Asset ─────────────────────────────────────────────────────
SYMBOL   = 'ETHUSDT'   # BTCUSDT | ETHUSDT | BNBUSDT | SOLUSDT
ENTRY_TF = '5m'        # 1m | 5m | 15m | 1h | 4h (5m for optimal entry)

# ── Prediction ────────────────────────────────────────────────
TRAIN_BARS = 200
FORECAST_H = 3

# ── Chronos ───────────────────────────────────────────────────
USE_CHRONOS = False  # Enable after torch install

# ── Risk ──────────────────────────────────────────────────────
ACCOUNT_BALANCE = 1000.0
MAX_RISK_PCT    = 0.01
RR_RATIO        = 2.0
ATR_SL_MULT     = 1.5

# ── Entry Optimization ────────────────────────────────────────
ENTRY_WAIT_BARS  = 3      # Wait up to 3 bars for optimal price
ENTRY_PRICE_RANGE = 0.002 # ~0.2% range for optimal entry (e.g., 1629-1631)

# ── Filters ───────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 55
ADX_TREND_THRESHOLD  = 25   # Crypto more volatile
RSI_OVERBOUGHT       = 75
RSI_OVERSOLD         = 25
USE_SMC_FILTER       = False  # Toggle SMC validation on/off

# ── TF Stack ──────────────────────────────────────────────────
_TF_MAP = {
    '1m':  ['1m',  '5m',  '15m', '1h'],
    '5m':  ['5m',  '15m', '1h',  '4h'],
    '15m': ['15m', '1h',  '4h',  '1d'],
    '1h':  ['1h',  '4h',  '1d'],
    '4h':  ['4h',  '1d'],
}
TIMEFRAMES = _TF_MAP.get(ENTRY_TF, [ENTRY_TF])

print(f'Asset:           {SYMBOL}')
print(f'Entry TF:        {ENTRY_TF}')
print(f'TF Stack:        {TIMEFRAMES}')
print(f'Train bars:      {TRAIN_BARS}')
print(f'Forecast:        {FORECAST_H} bars ahead')
print(f'SMC Filter:      {"ON" if USE_SMC_FILTER else "OFF"}')
print(f'Entry Optimize:  {ENTRY_WAIT_BARS} bars, ±{ENTRY_PRICE_RANGE*100:.1f}%')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 2a — MT5 AUTO-TRADING SETUP                  ║
# ╚══════════════════════════════════════════════════════════════╝

import MetaTrader5 as mt5

# Load credentials from .env
load_dotenv()
MT5_ACCOUNT = int(os.getenv('MT5_ACCOUNT'))
MT5_PASSWORD = os.getenv('MT5_PASSWORD')
MT5_SERVER = os.getenv('MT5_SERVER')
MT5_SYMBOL = os.getenv('MT5_SYMBOL', 'ETH')
MT5_LOT = float(os.getenv('MT5_LOT', '0.01'))

# MT5 Trading State
_mt5_connected = False
_consecutive_sl = 0
_stop_trading = False
_active_orders = {}

def init_mt5():
    """Initialize MT5 connection"""
    global _mt5_connected
    try:
        if not mt5.initialize(login=MT5_ACCOUNT,
                             password=MT5_PASSWORD,
                             server=MT5_SERVER):
            print(f'❌ MT5 init failed: {mt5.last_error()}')
            return False
        
        _mt5_connected = True
        account = mt5.account_info()
        print(f'✓ MT5 Connected: {MT5_ACCOUNT}@{MT5_SERVER}')
        print(f'  Balance: {account.balance:.2f} USD')
        print(f'  Equity: {account.equity:.2f} USD')
        return True
    except Exception as e:
        print(f'❌ MT5 error: {e}')
        return False


def place_trade(direction, entry, sl, tp1, tp2):
    """Place order with ladder"""
    global _consecutive_sl, _stop_trading, _active_orders
    
    if not _mt5_connected:
        print('❌ MT5 not connected')
        return None
    
    if _stop_trading:
        send_discord('🛑 **TRADING PAUSED** — 3 consecutive SL limit reached.')
        return None
    
    try:
        symbol_info = mt5.symbol_info(MT5_SYMBOL)
        if symbol_info is None:
            print(f'❌ Symbol {MT5_SYMBOL} not found')
            return None
        
        order_type = mt5.ORDER_TYPE_BUY if direction == 'BUY' else mt5.ORDER_TYPE_SELL
        
        request = {
            'action': mt5.TRADE_ACTION_DEAL,
            'symbol': MT5_SYMBOL,
            'volume': MT5_LOT,
            'type': order_type,
            'price': entry,
            'sl': sl,
            'tp': tp1,
            'deviation': 20,
            'magic': 0,
            'comment': f'{direction} @ {entry:.5f}',
            'type_time': mt5.ORDER_TIME_GTC,
        }
        
        result = mt5.order_send(request)
        
        if result.retcode != mt5.TRADE_RETCODE_DONE:
            print(f'❌ Order failed: {result.comment}')
            return None
        
        ticket = result.order
        _active_orders[ticket] = {
            'direction': direction,
            'entry': entry,
            'sl': sl,
            'tp1': tp1,
            'tp2': tp2,
            'status': 'open',
            'time': datetime.now(timezone.utc)
        }
        
        print(f'✅ Order placed: {ticket} | {direction} {MT5_LOT}L @ {entry:.5f}')
        return ticket
        
    except Exception as e:
        print(f'❌ Trade error: {e}')
        return None


def check_positions():
    """Check active positions for SL/TP hits"""
    global _consecutive_sl, _stop_trading, _active_orders
    
    if not _mt5_connected:
        return
    
    try:
        positions = mt5.positions_get()
        
        if not positions:
            _active_orders.clear()
            return
        
        for pos in positions:
            if pos.symbol != MT5_SYMBOL:
                continue
            
            ticket = pos.ticket
            if ticket not in _active_orders:
                continue
            
            # Check if position has profit/loss
            if pos.profit < -10:  # Stop loss hit
                _consecutive_sl += 1
                print(f'⛔ SL HIT: {ticket} | Loss: {pos.profit:.2f} | Count: {_consecutive_sl}/3')
                
                if _consecutive_sl >= 3:
                    _stop_trading = True
                    send_discord('🛑 **STOP TRADING** — 3 consecutive SL hits!')
                
                del _active_orders[ticket]
            elif pos.profit > 10:  # Take profit area
                _consecutive_sl = 0
                print(f'✅ TP HIT: {ticket} | Profit: {pos.profit:.2f}')
        
    except Exception as e:
        print(f'Position check error: {e}')


# Initialize MT5
print('\n' + '='*50)
print('MT5 AUTO-TRADING MODE')
print('='*50)
if init_mt5():
    print(f'Trading Symbol: {MT5_SYMBOL}')
    print(f'Lot Size: {MT5_LOT}')
    print(f'Stop Condition: 3 consecutive SL hits')
    print(f'Status: READY FOR SIGNALS ✓')
else:
    print('⚠️  MT5 not available')
print('='*50 + '\n')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 3 — DATA FEEDS                              ║
# ╚══════════════════════════════════════════════════════════════╝

def get_binance_bars(symbol='BTCUSDT',
                     interval='5m',
                     limit=500):
    """Fetch OHLCV from Binance — FREE, no API key"""
    try:
        url    = 'https://api.binance.com/api/v3/klines'
        params = {
            'symbol':   symbol,
            'interval': interval,
            'limit':    limit
        }
        r    = requests.get(url, params=params, timeout=15)
        data = r.json()

        if isinstance(data, dict) and 'code' in data:
            print(f"Binance error: {data['msg']}")
            return pd.DataFrame()

        df = pd.DataFrame(data, columns=[
            'timestamp', 'open', 'high', 'low',
            'close', 'volume', 'close_time',
            'quote_volume', 'trades',
            'taker_buy_base', 'taker_buy_quote', 'ignore'
        ])

        df['timestamp'] = pd.to_datetime(
            df['timestamp'], unit='ms', utc=True
        )
        df = df.set_index('timestamp')

        for col in ['open','high','low','close','volume']:
            df[col] = df[col].astype(float)

        return df[['open','high','low','close','volume']]

    except Exception as e:
        print(f"Data fetch error: {e}")
        return pd.DataFrame()


def get_current_price(symbol='BTCUSDT'):
    """Fetch current live price from Binance ticker (not candle close)"""
    try:
        url = 'https://api.binance.com/api/v3/ticker/price'
        params = {'symbol': symbol}
        r = requests.get(url, params=params, timeout=10)
        data = r.json()
        return float(data['price'])
    except Exception as e:
        print(f"Live price fetch error: {e}")
        return None


def get_multi_tf_crypto(symbol, timeframes, limit=500, verbose=False):
    """Fetch multiple timeframes"""
    data = {}
    for tf in timeframes:
        df = get_binance_bars(symbol, tf, limit)
        if not df.empty:
            data[tf] = df
            if verbose:
                print(f'  {tf:6s}: {len(df):4d} bars')
        else:
            if verbose:
                print(f'  {tf:6s}: FAILED')
    return data


# ── Technical Indicators ──────────────────────────────────────

def calc_atr(df, period=14):
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift(1)).abs(),
        (df['low']  - df['close'].shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()


def calc_rsi(close, period=14):
    delta = close.diff()
    gain  = delta.clip(lower=0).ewm(
        alpha=1/period, adjust=False
    ).mean()
    loss  = (-delta.clip(upper=0)).ewm(
        alpha=1/period, adjust=False
    ).mean()
    rs = gain / (loss + 1e-8)
    return 100 - (100 / (1 + rs))


def calc_adx(df, period=14):
    high  = df['high']
    low   = df['low']
    close = df['close']

    plus_dm  = high.diff()
    minus_dm = low.diff().abs() * -1

    plus_dm  = plus_dm.where(
        (plus_dm > 0) & (plus_dm > minus_dm.abs()), 0
    )
    minus_dm = minus_dm.abs().where(
        (minus_dm.abs() > 0) &
        (minus_dm.abs() > plus_dm), 0
    )

    tr  = calc_atr(df, period)
    pdi = 100 * plus_dm.ewm(
        alpha=1/period, adjust=False
    ).mean() / (tr + 1e-8)
    mdi = 100 * minus_dm.ewm(
        alpha=1/period, adjust=False
    ).mean() / (tr + 1e-8)
    dx  = 100 * (pdi - mdi).abs() / (pdi + mdi + 1e-8)
    adx = dx.ewm(alpha=1/period, adjust=False).mean()
    return adx


def compute_crypto_features(df):
    """Compute all technical features"""
    feat = pd.DataFrame(index=df.index)
    close = df['close']

    feat['rsi_14']    = calc_rsi(close, 14)
    feat['adx_14']    = calc_adx(df, 14)
    feat['atr_14']    = calc_atr(df, 14)

    ema12             = close.ewm(span=12).mean()
    ema26             = close.ewm(span=26).mean()
    macd              = ema12 - ema26
    feat['macd_hist'] = macd - macd.ewm(span=9).mean()

    feat['ema_20']    = close.ewm(span=20).mean()
    feat['ema_50']    = close.ewm(span=50).mean()

    bb_mid            = close.rolling(20).mean()
    bb_std            = close.rolling(20).std()
    feat['bb_pct']    = (
        (close - (bb_mid - 2*bb_std)) /
        (4*bb_std + 1e-8)
    ).clip(-0.1, 1.1)

    return feat


# ── Test Data ─────────────────────────────────────────────────
print(f'Fetching {SYMBOL} data...')
tf_data = get_multi_tf_crypto(SYMBOL, TIMEFRAMES, verbose=True)
print(f'✓ Data ready: {len(tf_data)} timeframes')

# Test live price
live_px = get_current_price(SYMBOL)
print(f'Current live price: {live_px}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 4 — ENSEMBLE FORECAST (WITH CHRONOS)       ║
# ╚══════════════════════════════════════════════════════════════╝

_SHARPE = {
    'AutoTheta': 0.771,
    'AutoETS':   0.162,
    'Chronos':   0.498
}
_W_SUM        = sum(_SHARPE.values())
MODEL_WEIGHTS = {k: v/_W_SUM for k, v in _SHARPE.items()}

print(f'Weights: '
      f'AutoTheta={MODEL_WEIGHTS["AutoTheta"]:.3f} '
      f'AutoETS={MODEL_WEIGHTS["AutoETS"]:.3f} '
      f'Chronos={MODEL_WEIGHTS["Chronos"]:.3f}')

# ── Chronos Loader ────────────────────────────────────────────
_chronos_forecaster = None

def _load_chronos():
    global _chronos_forecaster
    if not USE_CHRONOS:
        return None
    if _chronos_forecaster is not None:
        return _chronos_forecaster
    try:
        from chronos import ChronosPipeline
        import torch
        print('Loading Chronos-bolt-tiny...', end=' ')
        _chronos_forecaster = ChronosPipeline.from_pretrained(
            "amazon/chronos-bolt-tiny",
            device_map="cpu",
            torch_dtype=torch.float32
        )
        print('OK ✓')
    except Exception as e:
        print(f'Chronos failed ({e}) — using AutoTheta+AutoETS only')
        _chronos_forecaster = None
    return _chronos_forecaster


def _run_chronos(close_series, h):
    """Zero-shot Chronos prediction"""
    if not USE_CHRONOS:
        return None
    forecaster = _load_chronos()
    if forecaster is None:
        return None
    try:
        import torch
        context = torch.tensor(
            close_series.values[-200:].astype(float),
            dtype=torch.float32
        ).unsqueeze(0)

        with torch.no_grad():
            forecast = forecaster.predict(
                context=context,
                prediction_length=h,
                num_samples=20
            )

        median   = forecast[0].median(dim=0).values.numpy()
        last_px  = float(close_series.iloc[-1])
        log_rets = np.log(median / last_px)
        return log_rets

    except Exception as e:
        print(f'Chronos predict error: {e}')
        return None


def _season_len(tf):
    return {
        '1m': 390, '5m': 78, '15m': 26,
        '30m': 13, '1h': 7, '4h': 2, '1d': 5
    }.get(tf, 10)


def run_crypto_ensemble(df, tf, train_bars, h,
                        use_chronos=True):
    """AutoTheta + AutoETS + Chronos ensemble"""
    close   = df['close'].dropna()
    log_ret = np.log(close / close.shift(1)).dropna()
    series  = log_ret.iloc[-train_bars:].values.astype(float)
    n, sl   = len(series), _season_len(tf)

    # AutoTheta + AutoETS
    sf_df = pd.DataFrame({
        'unique_id': ['asset'] * n,
        'ds':        pd.date_range(
            '2000-01-01', periods=n, freq='15min'
        ),
        'y': series,
    })

    preds = StatsForecast(
        models=[
            AutoTheta(season_length=sl),
            AutoETS(season_length=sl)
        ],
        freq='15min', n_jobs=1
    ).forecast(df=sf_df, h=h)

    theta     = preds['AutoTheta'].values
    ets       = preds['AutoETS'].values
    theta_cum = np.cumsum(theta)
    ets_cum   = np.cumsum(ets)

    w_t = MODEL_WEIGHTS['AutoTheta']
    w_e = MODEL_WEIGHTS['AutoETS']
    w_c = MODEL_WEIGHTS['Chronos']

    # Chronos
    chronos_cum = None
    if use_chronos:
        chronos_log = _run_chronos(close, h)
        if chronos_log is not None:
            chronos_cum = chronos_log

    # Ensemble
    if chronos_cum is not None:
        w_sum   = w_t + w_e + w_c
        ens_cum = (
            w_t * theta_cum +
            w_e * ets_cum +
            w_c * chronos_cum
        ) / w_sum
        models_used = ['AutoTheta', 'AutoETS', 'Chronos']
    else:
        w_sum   = w_t + w_e
        ens_cum = (
            w_t * theta_cum +
            w_e * ets_cum
        ) / w_sum
        models_used = ['AutoTheta', 'AutoETS']

    pred_ret = float(ens_cum[0])
    last_px  = float(close.iloc[-1])
    pred_pxs = [
        last_px * np.exp(ens_cum[i]) for i in range(h)
    ]

    return {
        'direction':   int(np.sign(pred_ret)),
        'pred_return': pred_ret,
        'pred_prices': pred_pxs,
        'ensemble':    ens_cum,
        'models_used': models_used,
        'chronos_used': chronos_cum is not None,
    }


# Test
print('\nRunning test forecast...')
entry_df = tf_data[ENTRY_TF]
fc = run_crypto_ensemble(
    entry_df, ENTRY_TF, TRAIN_BARS,
    FORECAST_H, use_chronos=USE_CHRONOS
)
dir_label = (
    'BUY' if fc['direction'] > 0 else
    'SELL' if fc['direction'] < 0 else 'NEUTRAL'
)
print(f'Models:      {", ".join(fc["models_used"])}')
print(f'Direction:   {dir_label}')
print(f'Pred return: {fc["pred_return"]:+.5f}')

In [44]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 5 — MULTI-TF BIAS                          ║
# ╚══════════════════════════════════════════════════════════════╝

def score_crypto_tf(df):
    """Score a single timeframe for bias"""
    close = df['close']
    feat  = compute_crypto_features(df)

    score     = 0
    factors   = []
    direction = 0

    # EMA trend
    ema20 = feat['ema_20'].iloc[-1]
    ema50 = feat['ema_50'].iloc[-1]
    price = close.iloc[-1]

    if price > ema20:
        score += 1
        factors.append('price>ema20')
    if ema20 > ema50:
        score += 1
        factors.append('ema20>ema50')

    # MACD
    if feat['macd_hist'].iloc[-1] > 0:
        score += 1
        factors.append('macd_bull')
    else:
        score -= 1
        factors.append('macd_bear')

    # RSI
    rsi = feat['rsi_14'].iloc[-1]
    if rsi > 55:
        score += 1
        factors.append(f'rsi_bull({rsi:.0f})')
    elif rsi < 45:
        score -= 1
        factors.append(f'rsi_bear({rsi:.0f})')

    direction = 1 if score > 0 else (-1 if score < 0 else 0)
    strength  = abs(score) / 4

    return {
        'direction': direction,
        'strength':  strength,
        'factors':   factors,
        'score':     score
    }


# Multi-TF analysis
scores  = {tf: score_crypto_tf(df)
           for tf, df in tf_data.items()}
buy_tfs = [tf for tf, sc in scores.items()
           if sc['direction'] == 1]
sel_tfs = [tf for tf, sc in scores.items()
           if sc['direction'] == -1]

print(f'Multi-TF Bias for {SYMBOL}:')
print(f'{"TF":6s}  {"Direction":10s}  {"Strength":8s}')
print('-' * 40)
for tf, sc in scores.items():
    d = {1:'BULLISH', -1:'BEARISH', 0:'NEUTRAL'}[
        sc['direction']
    ]
    print(f'{tf:6s}  {d:10s}  {sc["strength"]:.2f}')

Multi-TF Bias for ETHUSDT:
TF      Direction   Strength
----------------------------------------
15m     BULLISH     0.25
1h      BEARISH     0.25
4h      NEUTRAL     0.00
1d      BEARISH     0.50


In [45]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 6 — NEWS + SENTIMENT                        ║
# ╚══════════════════════════════════════════════════════════════╝

def get_crypto_fear_greed():
    """Crypto specific Fear & Greed Index"""
    try:
        url = 'https://api.alternative.me/fng/'
        r   = requests.get(url, timeout=10)
        data = r.json()
        score  = int(data['data'][0]['value'])
        rating = data['data'][0]['value_classification']
        return score, rating
    except:
        return 50, 'Neutral'


def get_crypto_sentiment(direction):
    score, rating = get_crypto_fear_greed()

    if direction == 'BUY':
        if score < 40:
            # Crypto: Fear = BUY opportunity
            return (f'😨 Fear ({score}) '
                    f'= BUY opportunity ✓')
        elif score > 60:
            return (f'😎 Greed ({score}) '
                    f'— caution BUY')
        else:
            return f'😐 Neutral ({score})'
    else:
        if score > 60:
            return (f'😎 Greed ({score}) '
                    f'= SELL opportunity ✓')
        elif score < 40:
            return (f'😨 Fear ({score}) '
                    f'— caution SELL')
        else:
            return f'😐 Neutral ({score})'

def get_news_events():
    try:
        url = (
            "https://nfs.faireconomy.media/"
            "ff_calendar_thisweek.json"
        )
        r      = requests.get(url, timeout=10)
        events = r.json()
        now    = datetime.now(timezone.utc)

        upcoming = []
        for event in events:
            if event.get('impact') != 'High':
                continue
            try:
                et = datetime.fromisoformat(
                    event['date'].replace('Z', '+00:00')
                )
                diff = (et - now).total_seconds() / 60
                if -30 <= diff <= 60:
                    upcoming.append({
                        'title':   event.get('title', ''),
                        'country': event.get('country', ''),
                        'minutes': diff
                    })
            except:
                continue
        return upcoming
    except:
        return []


def is_news_zone():
    events   = get_news_events()
    relevant = ['USD', 'US', 'BTC', 'CRYPTO']
    for event in events:
        if event['country'] in relevant:
            return True, event
    return False, None


# Test
score, rating = get_crypto_fear_greed()
print(f'Crypto Fear & Greed: {score} ({rating})')

in_nz, nz_event = is_news_zone()
print(f'News zone: '
      f'{"⚠️ " + nz_event["title"] if in_nz else "Clear ✓"}')

Crypto Fear & Greed: 23 (Extreme Fear)
News zone: Clear ✓


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 7 — LADDER + SIGNAL                        ║
# ╚══════════════════════════════════════════════════════════════╝

def calculate_crypto_ladder(entry, direction, atr,
                             sl_mult=1.5, rr_ratio=2.0):
    """Calculate stop loss and take profit levels (SL1 removed)"""
    sl_dist  = atr * sl_mult
    tp1_dist = atr * 1.0
    tp2_dist = sl_dist * rr_ratio

    if direction == 'BUY':
        sl  = entry - sl_dist
        tp1 = entry + tp1_dist
        tp2 = entry + tp2_dist
    else:
        sl  = entry + sl_dist
        tp1 = entry - tp1_dist
        tp2 = entry - tp2_dist

    # Pips = price difference × 10 for crypto
    return {
        'sl':       round(sl, 2),
        'tp1':      round(tp1, 2),
        'tp2':      round(tp2, 2),
        'pips_sl':  round(sl_dist * 10, 0),
        'pips_tp1': round(tp1_dist * 10, 0),
        'pips_tp2': round(tp2_dist * 10, 0),
    }

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 7a — OPTIMAL ENTRY                          ║
# ╚══════════════════════════════════════════════════════════════╝

def calculate_optimal_entry(df, direction, wait_bars=3, 
                            price_range_pct=0.002):
    """Calculate optimal entry price within range"""
    if len(df) < wait_bars:
        current = df['close'].iloc[-1]
        return current, None
    
    recent = df.iloc[-wait_bars:]
    signal_price = df['close'].iloc[-1]
    
    if direction == 'BUY':
        optimal = recent['low'].min()
        range_low = signal_price * (1 - price_range_pct)
        range_high = signal_price
    else:  # SELL
        optimal = recent['high'].max()
        range_low = signal_price
        range_high = signal_price * (1 + price_range_pct)
    
    if direction == 'BUY':
        entry = max(optimal, range_low)
    else:
        entry = min(optimal, range_high)
    
    return entry, {
        'signal_price': signal_price,
        'optimal_price': optimal,
        'entry_price': entry,
        'range': (range_low, range_high)
    }

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 7b — SMC VALIDATION                         ║
# ║        (Structure Break + Order Block + FVG)               ║
# ╚══════════════════════════════════════════════════════════════╝

def has_broken_structure(df, lookback=20):
    """SMC: Check if price has broken recent swing high/low"""
    if len(df) < lookback:
        return False, None, None
    
    recent = df.iloc[-lookback:]
    swing_high = recent['high'].max()
    swing_low = recent['low'].min()
    current_close = df['close'].iloc[-1]
    
    broken = (current_close > swing_high or 
              current_close < swing_low)
    
    return broken, swing_high, swing_low


def detect_fvg(df, lookback=50):
    """Fair Value Gap detection: unfilled gaps between candles"""
    if len(df) < 3:
        return []
    
    fvgs = []
    recent = df.iloc[-lookback:].copy()
    
    for i in range(1, len(recent) - 1):
        prev_high = recent['high'].iloc[i-1]
        prev_low = recent['low'].iloc[i-1]
        curr_open = recent['open'].iloc[i]
        curr_close = recent['close'].iloc[i]
        next_low = recent['low'].iloc[i+1]
        next_high = recent['high'].iloc[i+1]
        
        # Bullish FVG: gap up (red→green)
        if curr_open > prev_high and next_low > prev_high:
            fvgs.append({
                'type': 'bullish',
                'bottom': prev_high,
                'top': min(curr_open, next_low),
                'index': i
            })
        
        # Bearish FVG: gap down (green→red)
        if curr_open < prev_low and next_high < prev_low:
            fvgs.append({
                'type': 'bearish',
                'top': prev_low,
                'bottom': max(curr_open, next_high),
                'index': i
            })
    
    return fvgs


def validate_entry_with_smc(df, current_price, signal_direction,
                            fresh_obs=20):
    """SMC Entry Validation: 3-point confirmation"""
    
    # Check 1: Structure Break
    broken, swing_h, swing_l = has_broken_structure(
        df, lookback=fresh_obs
    )
    if not broken:
        return False, "No structure break"
    
    # Check 2: Order Block Retest (price near recent OB)
    if len(df) >= fresh_obs + 5:
        ob_zone = df.iloc[-(fresh_obs+5):].copy()
        ob_prices = ob_zone['close'].values
        ob_mean = ob_prices.mean()
        ob_std = ob_prices.std()
        
        distance_from_ob = abs(
            current_price - ob_mean
        ) / (ob_std + 1e-8)
        
        if distance_from_ob > 2.5:
            return False, "Far from OB zone"
    
    # Check 3: Not inside FVG gap
    fvgs = detect_fvg(df, lookback=50)
    for fvg in fvgs:
        gap_low = min(fvg['bottom'], fvg['top'])
        gap_high = max(fvg['bottom'], fvg['top'])
        
        if gap_low < current_price < gap_high:
            if (fvg['type'] == 'bullish' and 
                signal_direction == -1):
                return False, "Inside bullish FVG"
            if (fvg['type'] == 'bearish' and 
                signal_direction == 1):
                return False, "Inside bearish FVG"
    
    return True, "SMC ✓"

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║           CELL 9 — LIVE SCANNER (MT5 + Discord)              ║
# ║           Auto-executes trades on MT5                        ║
# ╚══════════════════════════════════════════════════════════════╝

LIVE             = False
SCAN_INTERVAL_S  = 60
SCAN_WITH_TRADE  = 300

try:
    crypto_stop.set()
    time.sleep(2)
except:
    pass

crypto_stop = threading.Event()

_crypto_scanner_running = False

def _crypto_worker():
    global _crypto_scanner_running

    if _crypto_scanner_running:
        print("Already running — skipping")
        return

    _crypto_scanner_running = True
    _count            = 0
    _last             = None
    _last_signal_time = None
    _last_signal_dir  = None
    _trade_open       = False
    _trade_open_time  = None
    SIGNAL_COOLDOWN   = 2700
    TRADE_TIMEOUT     = 14400

    print(f"[CRYPTO] Scanner started — {SYMBOL}")
    print(f"No trade:   every {SCAN_INTERVAL_S}s")
    print(f"Trade open: every {SCAN_WITH_TRADE}s")

    while not crypto_stop.is_set():
        _count += 1
        now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')

        # Check active positions
        check_positions()

        # Auto-reset trade after 4 hours
        if _trade_open and _trade_open_time:
            age = (datetime.now(timezone.utc) - _trade_open_time).seconds
            if age > TRADE_TIMEOUT:
                _trade_open = False
                _trade_open_time = None
                print(f'[{now}] Trade auto-reset')

        current_interval = SCAN_WITH_TRADE if _trade_open else SCAN_INTERVAL_S

        try:
            fresh = get_multi_tf_crypto(SYMBOL, TIMEFRAMES, limit=500, verbose=False)
            if not fresh or ENTRY_TF not in fresh:
                print(f'[{now}] Data fetch failed')
                crypto_stop.wait(current_interval)
                continue

            f_entry = fresh[ENTRY_TF]
            f_price = get_current_price(SYMBOL)
            if f_price is None:
                f_price = float(f_entry['close'].iloc[-1])
            
            f_atr = float(calc_atr(f_entry, 14).iloc[-1])
            f_feat = compute_crypto_features(f_entry)

            f_fc = run_crypto_ensemble(f_entry, ENTRY_TF, TRAIN_BARS, FORECAST_H, use_chronos=USE_CHRONOS)
            bar_id = str(f_entry.index[-1])
            dir_lbl = ('BUY' if f_fc['direction'] > 0 else 'SELL' if f_fc['direction'] < 0 else 'FLAT')

            if dir_lbl != 'FLAT' and bar_id != _last:
                _last = bar_id
                f_adx = float(f_feat['adx_14'].iloc[-1])
                f_rsi = float(f_feat['rsi_14'].iloc[-1])
                f_macd = float(f_feat['macd_hist'].iloc[-1])
                f_ema20 = float(f_feat['ema_20'].iloc[-1])

                # Filters 1-5 (same as before)
                if f_adx < ADX_TREND_THRESHOLD:
                    print(f'[{now}][{_count:4d}] SKIP — ADX={f_adx:.1f}')
                    continue
                if dir_lbl == 'BUY' and f_rsi > RSI_OVERBOUGHT:
                    print(f'[{now}][{_count:4d}] SKIP BUY RSI={f_rsi:.0f}')
                    continue
                if dir_lbl == 'SELL' and f_rsi < RSI_OVERSOLD:
                    print(f'[{now}][{_count:4d}] SKIP SELL RSI={f_rsi:.0f}')
                    continue
                
                if (_last_signal_time and dir_lbl == _last_signal_dir):
                    elapsed = (datetime.now(timezone.utc) - _last_signal_time).seconds
                    if elapsed < SIGNAL_COOLDOWN:
                        print(f'[{now}][{_count:4d}] SKIP cooldown {elapsed}s')
                        continue

                f_scores = {tf: score_crypto_tf(df) for tf, df in fresh.items()}
                aligned = [tf for tf, sc in f_scores.items() if sc['direction'] == (1 if dir_lbl == 'BUY' else -1)]
                opposite = [tf for tf, sc in f_scores.items() if sc['direction'] == (-1 if dir_lbl == 'BUY' else 1)]
                total_tfs = len(f_scores)
                align_pct = (len(aligned) / total_tfs * 100 if total_tfs > 0 else 0)

                if len(aligned) < 2:
                    print(f'[{now}][{_count:4d}] SKIP TF {len(aligned)}/{total_tfs}')
                    continue

                conf = 0
                conf += min(35, int(align_pct * 0.5))
                conf += (20 if f_adx >= ADX_TREND_THRESHOLD else 5)
                conf += (15 if abs(f_fc['pred_return']) > 0.0003 else 8)
                conf += (15 if (f_macd > 0) == (dir_lbl == 'BUY') else 0)
                conf += (10 if (f_price > f_ema20) == (dir_lbl == 'BUY') else 0)
                conf += 5 if len(opposite) == 0 else 0
                conf = min(conf, 95)

                if conf < CONFIDENCE_THRESHOLD:
                    print(f'[{now}][{_count:4d}] SKIP conf={conf}%')
                    continue

                smc_tag = ''
                if USE_SMC_FILTER:
                    signal_dir_int = 1 if dir_lbl == 'BUY' else -1
                    smc_valid, smc_reason = validate_entry_with_smc(f_entry, f_price, signal_dir_int, fresh_obs=20)
                    if not smc_valid:
                        print(f'[{now}][{_count:4d}] SKIP — SMC: {smc_reason}')
                        continue
                    smc_tag = ' | SMC=✓'

                entry_price, entry_info = calculate_optimal_entry(f_entry, dir_lbl, ENTRY_WAIT_BARS, ENTRY_PRICE_RANGE)

                tp_prob = (70 if f_adx >= 40 else 58 if f_adx >= 30 else 48)
                if abs(f_fc['pred_return']) >= 0.0003:
                    tp_prob += 10
                tp_prob = min(tp_prob, 85)

                ladder = calculate_crypto_ladder(entry_price, dir_lbl, f_atr)

                in_nz, nz_event = is_news_zone()
                news_warning = (f'⚠️ NEWS: {nz_event["title"]} in {nz_event["minutes"]:.0f} min' if in_nz else '📰 News: Clear')
                sentiment_warning = get_crypto_sentiment(dir_lbl)

                _last_signal_time = datetime.now(timezone.utc)
                _last_signal_dir = dir_lbl
                _trade_open = True
                _trade_open_time = datetime.now(timezone.utc)

                # ── MT5 AUTO EXECUTION ──────────────────
                mt5_status = '❌ NOT CONNECTED'
                order_id = None
                if _mt5_connected:
                    order_id = place_trade(dir_lbl, entry_price, ladder['sl'], ladder['tp1'], ladder['tp2'])
                    if order_id:
                        mt5_status = f'✅ EXECUTED (#{order_id})'
                    else:
                        mt5_status = '⚠️ EXECUTION FAILED'

                coin = SYMBOL.replace('USDT', '')
                entry_detail = ''
                if entry_info:
                    entry_detail = (f'\n📊 Entry Strategy: Wait {ENTRY_WAIT_BARS} bars\n'
                                  f'Range: {entry_info["range"][0]:.2f} - {entry_info["range"][1]:.2f}\n'
                                  f'Optimal Entry: {entry_price:.2f}')
                
                msg = (f'🎯 **NEW SIGNAL** {dir_lbl} {coin}/USDT{smc_tag}\n'
                      f'━━━━━━━━━━━━━━━━━━━━━━\n'
                      f'📅 {datetime.now(timezone.utc).strftime("%A, %d %b %Y")}\n'
                      f'🕐 {now}\n'
                      f'💰 Live Price: {f_price:.2f}\n'
                      f'{entry_detail}\n'
                      f'━━━━━━━━━━━━━━━━━━━━━━\n'
                      f'**Risk Management:**\n'
                      f'SL: {ladder["sl"]:.2f} (-{ladder["pips_sl"]:.0f} pips)\n'
                      f'TP1: {ladder["tp1"]:.2f} (+{ladder["pips_tp1"]:.0f} pips)\n'
                      f'TP2: {ladder["tp2"]:.2f} (+{ladder["pips_tp2"]:.0f} pips)\n'
                      f'━━━━━━━━━━━━━━━━━━━━━━\n'
                      f'**Analytics:**\n'
                      f'Confidence: {conf}% | TP Hit: {tp_prob}% | ADX: {f_adx:.1f}\n'
                      f'RSI: {f_rsi:.1f} | TF Align: {len(aligned)}/{total_tfs}\n'
                      f'Prediction: {f_fc["pred_return"]:+.5f}\n'
                      f'━━━━━━━━━━━━━━━━━━━━━━\n'
                      f'{news_warning}\n'
                      f'{sentiment_warning}\n'
                      f'**MT5 Execution:** {mt5_status}\n'
                      f'━━━━━━━━━━━━━━━━━━━━━━\n'
                      f'🔬 Beta Testing — Ahmed R. Hussain')

                print(msg)
                send_discord(msg)

            else:
                if _count % 3 == 0:
                    status = ('TRADE OPEN' if _trade_open else 'Ready')
                    mt5_status = '✅' if _mt5_connected else '❌'
                    print(f'[{now}][{_count:4d}] [{SYMBOL}] {dir_lbl:4s} | Price: {f_price:.2f} | MT5: {mt5_status} | {status}')

        except Exception as exc:
            print(f'[{now}] Error: {exc}')

        crypto_stop.wait(current_interval)

    _crypto_scanner_running = False
    print('[CRYPTO] Scanner stopped.')


# ── Start ──────────────────────────────────────────────────────
if LIVE:
    crypto_stop.clear()
    t = threading.Thread(target=_crypto_worker, name='crypto_scanner', daemon=True)
    t.start()
    print(f'[CRYPTO] Scanner ON: {SYMBOL} {ENTRY_TF}')
    print(f'Scan every: {SCAN_INTERVAL_S}s')
    print('Set LIVE=False and re-run to stop.')
else:
    crypto_stop.set()
    print('[CRYPTO] Scanner OFF.')